## Module 2: Robust Verification on HumanEval

This notebook explores various verification techniques as presented in Module two of Stanford's CS329A Course (Self improving agents.) The question we will explore is: **if an LLM generates several candidate programs, how well can we pick the correct one?**

We keep generation and verification separate:

- generate candidate solutions
- freeze the candidate pool
- try several selection methods
- compare results and inspect failures

### Notebook Structure

**Part 1 (Baseline):** Generate 1 candidate per task → evaluate → done (not reused)

**Part 2 (Build pool):** Generate 5 candidates per task → **freeze this pool**

**Parts 3–5 (Selection strategies):** Apply different selection methods to the **same frozen pool** from Part 2:
- Part 3: Oracle (theoretical best-case)
- Part 4: Public tests (realistic weak verifier)
- Part 5: LLM judge (language-based reasoning)

**Part 6 (Failure analysis):** Analyze where each selection method helps or fails using the same pool

### Key Insight

By fixing the candidate pool, performance differences come from **how well we select**, not **what we generate**.


## Part 0 - Setup & EDA

Import libraries and load helper utilities.


In [1]:
# Core imports
import gc
import json
import math
import random
import re
import textwrap
import time
from pathlib import Path

import pandas as pd
from datasets import load_dataset
from IPython.display import display

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

pd.set_option("display.max_colwidth", 200)
notebook_start_time = time.time()


## HumanEval Dataset

**HumanEval** is a code benchmark. Each task gives a function prompt, then checks the generated code with Python assertions.

Key columns: `task_id`, `prompt`, `entry_point`, `test`, and `canonical_solution`.

A solution passes only if it runs and all tests succeed.


In [2]:
print("Loading HumanEval dataset...")
dataset = load_dataset("openai_humaneval")
print("Done.\n")
print("Available splits:", list(dataset.keys()))
for split in dataset.keys():
    print(f"{split}: {len(dataset[split])} examples")

problems_df = dataset["test"].to_pandas()
print("\nDataFrame shape:", problems_df.shape)


Loading HumanEval dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Done.

Available splits: ['test']
test: 164 examples

DataFrame shape: (164, 5)


We can view the dataset as a dataframe to better understand the structure.


In [3]:
problems_df.head()

,task_id,prompt,canonical_solution,test,entry_point
0,HumanEval/0,"from typing import List\n\n\ndef has_close_elements(numbers: List[float], threshold: float) -> bool:\n """""" Check if in given list of numbers, are any two numbers closer to each other than\n ...","for idx, elem in enumerate(numbers):\n for idx2, elem2 in enumerate(numbers):\n if idx != idx2:\n distance = abs(elem - elem2)\n if distance < t...","\n\nMETADATA = {\n 'author': 'jt',\n 'dataset': 'test'\n}\n\n\ndef check(candidate):\n assert candidate([1.0, 2.0, 3.9, 4.0, 5.0, 2.2], 0.3) == True\n assert candidate([1.0, 2.0, 3.9, ...",has_close_elements
1,HumanEval/1,"from typing import List\n\n\ndef separate_paren_groups(paren_string: str) -> List[str]:\n """""" Input to this function is a string containing multiple groups of nested parentheses. Your goal is t...",result = []\n current_string = []\n current_depth = 0\n\n for c in paren_string:\n if c == '(':\n current_depth += 1\n current_string.append(c)\n e...,"\n\nMETADATA = {\n 'author': 'jt',\n 'dataset': 'test'\n}\n\n\ndef check(candidate):\n assert candidate('(()()) ((())) () ((())()())') == [\n '(()())', '((()))', '()', '((())()())'...",separate_paren_groups
2,HumanEval/2,"\n\ndef truncate_number(number: float) -> float:\n """""" Given a positive floating point number, it can be decomposed into\n and integer part (largest integer smaller than given number) and de...",return number % 1.0\n,"\n\nMETADATA = {\n 'author': 'jt',\n 'dataset': 'test'\n}\n\n\ndef check(candidate):\n assert candidate(3.5) == 0.5\n assert abs(candidate(1.33) - 0.33) < 1e-6\n assert abs(candidat...",truncate_number
3,HumanEval/3,"from typing import List\n\n\ndef below_zero(operations: List[int]) -> bool:\n """""" You're given a list of deposit and withdrawal operations on a bank account that starts with\n zero balance. ...",balance = 0\n\n for op in operations:\n balance += op\n if balance < 0:\n return True\n\n return False\n,"\n\nMETADATA = {\n 'author': 'jt',\n 'dataset': 'test'\n}\n\n\ndef check(candidate):\n assert candidate([]) == False\n assert candidate([1, 2, -3, 1, 2, -3]) == False\n assert candi...",below_zero
4,HumanEval/4,"from typing import List\n\n\ndef mean_absolute_deviation(numbers: List[float]) -> float:\n """""" For a given list of input numbers, calculate Mean Absolute Deviation\n around the mean of this ...",mean = sum(numbers) / len(numbers)\n return sum(abs(x - mean) for x in numbers) / len(numbers)\n,"\n\nMETADATA = {\n 'author': 'jt',\n 'dataset': 'test'\n}\n\n\ndef check(candidate):\n assert abs(candidate([1.0, 2.0, 3.0]) - 2.0/3.0) < 1e-6\n assert abs(candidate([1.0, 2.0, 3.0, 4....",mean_absolute_deviation


We can also inspect one row to see the structure of our prompt and tests.


In [4]:
row = problems_df.iloc[0]
print("TASK ID:", row["task_id"])
print("\nENTRY POINT:", row["entry_point"])
print("\nPROMPT:\n")
print(row["prompt"])
print("\nTEST CODE:\n")
print(row["test"])
print("\nCANONICAL SOLUTION:\n")
print(row["canonical_solution"])


TASK ID: HumanEval/0

ENTRY POINT: has_close_elements

PROMPT:

from typing import List


def has_close_elements(numbers: List[float], threshold: float) -> bool:
    """ Check if in given list of numbers, are any two numbers closer to each other than
    given threshold.
    >>> has_close_elements([1.0, 2.0, 3.0], 0.5)
    False
    >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)
    True
    """


TEST CODE:



METADATA = {
    'author': 'jt',
    'dataset': 'test'
}


def check(candidate):
    assert candidate([1.0, 2.0, 3.9, 4.0, 5.0, 2.2], 0.3) == True
    assert candidate([1.0, 2.0, 3.9, 4.0, 5.0, 2.2], 0.05) == False
    assert candidate([1.0, 2.0, 5.9, 4.0, 5.0], 0.95) == True
    assert candidate([1.0, 2.0, 5.9, 4.0, 5.0], 0.8) == False
    assert candidate([1.0, 2.0, 3.0, 4.0, 5.0, 2.0], 0.1) == True
    assert candidate([1.1, 2.2, 3.1, 4.1, 5.1], 1.0) == True
    assert candidate([1.1, 2.2, 3.1, 4.1, 5.1], 0.5) == False



CANONICAL SOLUTION:

    for idx, elem in

Reusable evaluation code lives in `helpers.py` and is imported as `hh`.

Later cells assign helpers to short local names, such as `run_baseline = hh.run_baseline`, so the notebook stays readable.


In [5]:
# Load helpers from this module directory
import importlib.util
from pathlib import Path

module_dir = Path.cwd()
if not (module_dir / "helpers.py").exists():
    module_dir = Path("modules/02_robust_verification")

helpers_path = module_dir / "helpers.py"
if not helpers_path.exists():
    raise FileNotFoundError(f"Could not locate {helpers_path}")

spec = importlib.util.spec_from_file_location("rv_helpers", helpers_path)
if spec is None or spec.loader is None:
    raise ImportError(f"Could not create an import spec for {helpers_path}")
hh = importlib.util.module_from_spec(spec)
spec.loader.exec_module(hh)


## Fixed Parameters

Core settings shared across experiments.


In [6]:
MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"  # LLM used for code generation
JUDGE_MODEL_NAME = "Qwen/Qwen2.5-Coder-3B-Instruct"  # Stronger LLM used only for judging
MAX_TASKS = 50 # Number of HumanEval problems to evaluate (subset for speed)
NUM_CANDIDATES = 5  # Number of solutions generated per task
K_VALUES = [1, 2, NUM_CANDIDATES]  # Values of k for pass@k evaluation

MAX_NEW_TOKENS = 512  # Max tokens generated per solution
EXEC_TIMEOUT_SECONDS = 5  # Time limit for executing generated code
PUBLIC_TEST_FRACTION = 0.4  # Fraction of tests used for weak verification
RUN_LLM_JUDGE = True  # Whether to use an LLM to rank candidate solutions

RANDOM_SEED = 7  # Seed for reproducibility (sampling + dataset shuffling)
GENERATION_TOP_P = 0.95  # Nucleus sampling parameter for diversity

JUDGE_MAX_CANDIDATE_CHARS = 1600  # Max length of candidate code passed to judge
JUDGE_MAX_NEW_TOKENS = 96  # Max tokens generated by the LLM judge

SYSTEM_PROMPT = (
    "You complete Python functions for programming benchmarks. "
    "Return only valid Python code. Do not use markdown fences. Do not explain your answer."
)  # Instruction prompt guiding model behavior

## Model Initialization

Use a small instruction-tuned code model so the experiment can run locally while still showing generation and verification effects.


In [7]:
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

print("Random seed:", RANDOM_SEED)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype="auto",
)
model.eval()

print("Loaded model:", MODEL_NAME)


Random seed: 7
CUDA available: True
GPU: Tesla T4


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loaded model: Qwen/Qwen2.5-Coder-1.5B-Instruct


## Prompting & Candidate Generation

Candidate generation uses the same prompt format for both the baseline and the multi-sample pool. The difference is the sampling setting:

- `temperature = 0.0`: used for the deterministic baseline, where we want one stable first answer.
- `temperature > 0`: used for the candidate pool, where we want diverse solutions so `pass@k` can measure whether any sampled answer is correct.

This separation is important for the verification experiment. Generation creates possible answers, then later sections freeze those candidates and compare different ways to select the best one.


In [8]:
eval_problems_df = problems_df.iloc[:MAX_TASKS].copy().reset_index(drop=True)
print("Evaluation subset size:", len(eval_problems_df))


def build_messages(problem):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": (
                "Complete the following HumanEval Python function. "
                "Return only the code continuation needed to finish the function.\n\n"
                f"{problem['prompt']}"
            ),
        },
    ]


def generate_candidates(
    problem,
    num_samples=1,
    temperature=0.0,
    max_new_tokens=MAX_NEW_TOKENS,
    top_p=GENERATION_TOP_P,
):
    prompt_text = tokenizer.apply_chat_template(
        build_messages(problem),
        tokenize=False,
        add_generation_prompt=True,
    )

    return hh.generate_candidates(
        model,
        tokenizer,
        prompt_text,
        num_samples=num_samples,
        temperature=temperature,
        max_new_tokens=max_new_tokens,
        top_p=top_p,
    )

Evaluation subset size: 50


## End-to-End Debug Example

Run one task first to confirm prompting, parsing, and execution all work before launching the full experiment.


In [9]:
# Debug one HumanEval example end to end using the helper function

debug_example = hh.debug_example

debug_info = debug_example(
    problem=eval_problems_df.iloc[0].to_dict(),
    build_messages_fn=build_messages,
    generate_fn=generate_candidates,
    tokenizer=tokenizer,
    max_new_tokens=MAX_NEW_TOKENS,
    timeout_seconds=EXEC_TIMEOUT_SECONDS,
    temperature=0.0,
)

function_source = hh.assemble_candidate_source(
    prompt=debug_info["problem"]["prompt"],
    completion=debug_info["raw_completion"],
    entry_point=debug_info["problem"]["entry_point"],
)
print("\nModel-generated function:")
print(function_source)

print("\nDebug summary:")
print(json.dumps(debug_info["evaluation"], indent=2))


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


TASK ID: HumanEval/0
ENTRY POINT: has_close_elements

ORIGINAL HUMANEVAL PROMPT
from typing import List


def has_close_elements(numbers: List[float], threshold: float) -> bool:
    """ Check if in given list of numbers, are any two numbers closer to each other than
    given threshold.
    >>> has_close_elements([1.0, 2.0, 3.0], 0.5)
    False
    >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)
    True
    """


CHAT PROMPT SENT TO MODEL
<|im_start|>system
You complete Python functions for programming benchmarks. Return only valid Python code. Do not use markdown fences. Do not explain your answer.<|im_end|>
<|im_start|>user
Complete the following HumanEval Python function. Return only the code continuation needed to finish the function.

from typing import List


def has_close_elements(numbers: List[float], threshold: float) -> bool:
    """ Check if in given list of numbers, are any two numbers closer to each other than
    given threshold.
    >>> has_close_elements([1

The example output behaved as expected: the generated code for `HumanEval/0` passed the full HumanEval checks for that task.

## Part 1 - Baseline (pass@1)

Generate one answer per task and score it on the full HumanEval tests.

This gives the simplest comparison point before using extra candidates or smarter selection.


In [10]:
run_baseline = hh.run_baseline

baseline_metrics, baseline_rows = run_baseline(
    problems_df=eval_problems_df,
    generate_fn=generate_candidates,
    max_tasks=None,
    temperature=0.0,
    max_new_tokens=MAX_NEW_TOKENS,
    timeout_seconds=EXEC_TIMEOUT_SECONDS,
)

print(json.dumps(baseline_metrics, indent=2))


Baseline pass@1:   0%|          | 0/50 [00:00<?, ?it/s]

{
  "method": "baseline_pass@1",
  "num_tasks": 50,
  "num_passed": 34,
  "pass@1": 0.68,
  "syntax_errors": 0,
  "syntax_error_rate": 0.0,
  "runtime_failures": 16,
  "runtime_failure_rate": 0.32,
  "status_counts": {
    "passed": 34,
    "runtime_error": 5,
    "failed_tests": 11
  }
}


In [11]:
select_failure_examples = hh.select_failure_examples

baseline_metrics_df = pd.DataFrame([baseline_metrics])
baseline_failures_df = select_failure_examples(baseline_rows, limit=5)

display(baseline_metrics_df)
display(baseline_failures_df)


,method,num_tasks,num_passed,pass@1,syntax_errors,syntax_error_rate,runtime_failures,runtime_failure_rate,status_counts
0,baseline_pass@1,50,34,0.68,0,0.0,16,0.32,"{'passed': 34, 'runtime_error': 5, 'failed_tests': 11}"


,task_id,status,error_type,error_message,raw_completion
0,HumanEval/5,runtime_error,NameError,name 'delimiter' is not defined,```python result = [] for i in range(len(numbers)): result.append(numbers[i]) if i < len(numbers) - 1: result.append(delimiter) return result ```
1,HumanEval/6,failed_tests,AssertionError,One or more tests failed,```python stack = [] max_depth = 0 for char in paren_string: if char == '(': stack.append(char) depth = len(stack) if depth > max_depth: max_depth = depth elif char == ')': ...
2,HumanEval/8,runtime_error,NameError,name 'reduce' is not defined,"```python total_sum = sum(numbers) total_product = 1 if len(numbers) == 0 else reduce(lambda x, y: x * y, numbers) return total_sum, total_product ```"
3,HumanEval/10,runtime_error,IndexError,string index out of range,```python # Step 1: Find the longest palindromic suffix i = len(string) while i > 0 and string[i-1] != string[i]: i -= 1 # Step 2: Reverse the prefix that comes before the ...
4,HumanEval/17,runtime_error,ValueError,Invalid character encountered,```python result = [] i = 0 while i < len(music_string): if music_string[i] == 'o': result.append(4) i += 1 elif music_string[i:i+2] == 'o|': result.append(2) i += 2 elif ...


### Baseline Results

The single-sample baseline solved **34 of 50 tasks**, so `pass@1 = 0.68`. There were **no syntax errors**, but **16 tasks** still failed: **5 runtime errors** and **11 failed tests**.

This asks: "If we accept the first completion, how often is it right?" `pass@k` asks a different question: "If we sample several completions, does at least one work?"


## Part 2 - Test-Time Compute (pass@k)

Now we generate several candidates per task and freeze that pool. Every later method selects from the same candidates.

`pass@k` measures whether at least one of the first `k` candidates solves the task.


In [12]:
build_candidate_pool = hh.build_candidate_pool

candidate_pool_df = build_candidate_pool(
    problems_df=eval_problems_df,
    generate_fn=generate_candidates,
    num_candidates=NUM_CANDIDATES,
    max_tasks=None,
    temperature=0.8,
    max_new_tokens=MAX_NEW_TOKENS,
    timeout_seconds=EXEC_TIMEOUT_SECONDS,
)

print("Candidate pool shape:", candidate_pool_df.shape)
display(candidate_pool_df.head())


Generating 5 candidates:   0%|          | 0/50 [00:00<?, ?it/s]

Candidate pool shape: (250, 10)


,task_id,entry_point,passed,status,error_type,error_message,traceback,program,raw_completion,sample_index
0,HumanEval/0,has_close_elements,True,passed,None,None,None,"from typing import List\n\n\ndef has_close_elements(numbers: List[float], threshold: float) -> bool:\n """""" Check if in given list of numbers, are any two numbers closer to each other than\n ...","```python\nfor i in range(len(numbers)):\n for j in range(i + 1, len(numbers)):\n if abs(numbers[i] - numbers[j]) < threshold:\n return True\nreturn False\n```",0
1,HumanEval/0,has_close_elements,False,runtime_error,NameError,name 'combinations' is not defined,NameError: name 'combinations' is not defined,"from typing import List\n\n\ndef has_close_elements(numbers: List[float], threshold: float) -> bool:\n """""" Check if in given list of numbers, are any two numbers closer to each other than\n ...","```python\nreturn any(abs(a - b) < threshold for a, b in combinations(numbers, 2))\n```",1
2,HumanEval/0,has_close_elements,True,passed,None,None,None,"from typing import List\n\n\ndef has_close_elements(numbers: List[float], threshold: float) -> bool:\n """""" Check if in given list of numbers, are any two numbers closer to each other than\n ...","```python\n for i in range(len(numbers)):\n for j in range(i + 1, len(numbers)):\n if abs(numbers[i] - numbers[j]) < threshold:\n return True\n return False\...",2
3,HumanEval/0,has_close_elements,False,runtime_error,NameError,name 'combinations' is not defined,NameError: name 'combinations' is not defined,"from typing import List\n\n\ndef has_close_elements(numbers: List[float], threshold: float) -> bool:\n """""" Check if in given list of numbers, are any two numbers closer to each other than\n ...","```\n return any(abs(a - b) < threshold for a, b in combinations(numbers, 2))\n```",3
4,HumanEval/0,has_close_elements,True,passed,None,None,None,"from typing import List\n\n\ndef has_close_elements(numbers: List[float], threshold: float) -> bool:\n """""" Check if in given list of numbers, are any two numbers closer to each other than\n ...","for i in range(len(numbers)):\n for j in range(i + 1, len(numbers)):\n if abs(numbers[i] - numbers[j]) < threshold:\n return True\n return False",4


In [13]:
pass_at_k_table = hh.pass_at_k_table

passk_df = pass_at_k_table(candidate_pool_df, K_VALUES)
status_counts_df = candidate_pool_df["status"].value_counts(dropna=False).rename_axis("status").reset_index(name="count")

display(passk_df)
display(status_counts_df)


,k,num_tasks,empirical_pass@k,estimated_pass@k
0,1,50,0.58,0.58
1,2,50,0.68,0.71
2,5,50,0.80,0.80


,status,count
0,passed,145
1,failed_tests,61
2,runtime_error,39
3,syntax_error,5


### Interpreting pass@k

The frozen pool's **Pass@1 is 0.58**, below the baseline's **0.68**, because the pool uses sampled generation rather than the exact deterministic baseline setup.

The key result is **Pass@5 = 0.80**: by five samples, **40 of 50 tasks** have a correct answer somewhere. More samples help, but a verifier still has to pick the right candidate.


## Part 3 - Oracle vs Deployable Selection

The oracle answers: "If we could see the hidden test results, how often would we choose a correct candidate?"

It is useful as a ceiling, but not deployable, because real systems cannot use hidden evaluation tests to choose answers. That makes it an **upper bound**, not a real verifier.


In [14]:
oracle_best_of_n = hh.oracle_best_of_n
summary_table = hh.summary_table

oracle_metrics = oracle_best_of_n(candidate_pool_df, n=NUM_CANDIDATES)
summary_df = summary_table(baseline_metrics, passk_df, oracle_metrics)

print(json.dumps(oracle_metrics, indent=2))
display(summary_df)


{
  "method": "oracle_best_of_5",
  "num_tasks": 50,
  "oracle_accuracy": 0.8,
  "deployable": false,
  "reason": "Selection uses hidden evaluation tests to identify the winning candidate, which is only valid as an upper bound."
}


,method,metric,value,notes
0,Baseline,pass@1,0.68,One completion per task
1,Pass@1,empirical_pass@k,0.58,First sampled candidate from the frozen pool; not identical to the baseline if temperatures differ
2,Pass@2,empirical_pass@k,0.68,At least one passing candidate among first k samples
3,Pass@5,empirical_pass@k,0.80,At least one passing candidate among first k samples
4,oracle_best_of_5,oracle_accuracy,0.80,"Selection uses hidden evaluation tests to identify the winning candidate, which is only valid as an upper bound."


Oracle accuracy is **0.80**, the same as **Pass@5 = 0.80**. That is expected: `pass@5` says a correct candidate exists, and the oracle uses hidden tests to pick it.

The remaining **10 of 50 tasks** need better generation, more samples, or repair because none of the five candidates passed.


## Part 4 - Weak Verifier: Public Tests

A **verifier** helps choose between generated answers. Instead of trusting the first completion, we generate several candidates and ask which one has the strongest evidence of being correct.

Common verifier examples:

- **Executable tests**: run code and check outputs
- **Rule-based checks**: check format or constraints
- **Model judges**: ask another LLM to choose
- **Reward models**: score candidates with a trained model
- **Oracle verifiers**: use hidden answers or hidden tests, useful only for analysis

In this section, we use an **executable-test verifier** from `helpers.py`. It splits HumanEval's tests into a simulated public/private split, runs each frozen-pool candidate on the public chunks, and chooses the candidate with the best `public_score`.

This mirrors the weak-verifier setup from the module: generate multiple answers, then use a cheaper or partial signal to select one. Related examples include [**Shrinking the Generation-Verification Gap with Weak Verifiers**](https://arxiv.org/abs/2506.18203) and [**Training Verifiers to Solve Math Word Problems**](https://arxiv.org/abs/2110.14168).


In [15]:
evaluate_public_test_verifier = hh.evaluate_public_test_verifier

public_metrics, public_selection_df = evaluate_public_test_verifier(
    problems_df=eval_problems_df,
    pool_df=candidate_pool_df,
    public_fraction=PUBLIC_TEST_FRACTION,
    timeout_seconds=EXEC_TIMEOUT_SECONDS,
)

print(json.dumps(public_metrics, indent=2))
print("Tasks with zero public cases:", int((public_selection_df["num_public_cases"] == 0).sum()))
display(public_selection_df.head())


{
  "method": "public_test_verifier",
  "num_tasks": 50,
  "accuracy": 0.76,
  "avg_public_score": 0.84,
  "avg_public_cases": 2.24
}
Tasks with zero public cases: 3


,task_id,selected_sample_index,selected_passed_full_tests,selected_status,public_score,public_pass_count,public_total,oracle_available,num_candidates,num_public_cases,num_total_cases
0,HumanEval/0,0,True,passed,1.0,3,3,True,5,3,7
1,HumanEval/1,0,True,passed,1.0,2,2,True,5,2,4
2,HumanEval/10,2,True,passed,1.0,2,2,True,5,2,5
3,HumanEval/11,0,True,passed,1.0,2,2,True,5,2,3
4,HumanEval/12,1,False,failed_tests,1.0,2,2,True,5,2,3


The table shows what the weak verifier chose after scoring candidates with public tests.

The public-test verifier reached **0.76 accuracy**, meaning its selected candidate passed the full evaluator on **38 of 50 tasks**.

Useful columns:

- `selected_sample_index`: which candidate the verifier chose
- `public_score`: fraction of public test chunks the chosen candidate passed
- `public_pass_count` / `public_total`: the raw count behind `public_score`
- `selected_passed_full_tests`: whether the chosen candidate passed the full evaluator
- `oracle_available`: whether any candidate in the pool was correct

The important caveat: a high `public_score` is useful evidence, not proof. `HumanEval/12` shows this because the selected candidate passed the public tests but failed the full evaluator.


## Part 5 - LLM as a Judge

Here the selector reads the prompt and candidate code from the same frozen candidate pool, but does not execute anything.

For this section, the notebook loads a separate judge model using `JUDGE_MODEL_NAME`. Using a stronger judge than the generator is common because the judge may catch mistakes the generator missed. The default judge is stronger than the generator but still sized for a typical notebook GPU. If you have more memory, you can try `Qwen/Qwen2.5-Coder-7B-Instruct`; if memory is limited, set `JUDGE_MODEL_NAME = MODEL_NAME` to reuse the generator model.


In [16]:
make_llm_judge_fn = hh.make_llm_judge_fn

if RUN_LLM_JUDGE:
    if JUDGE_MODEL_NAME == MODEL_NAME:
        print("Reusing generator model as LLM judge:", JUDGE_MODEL_NAME)
        judge_tokenizer = tokenizer
        judge_model = model
    else:
        print("Loading separate LLM judge model:", JUDGE_MODEL_NAME)
        # Candidate generation is complete, so free the generator model before loading the judge.
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        judge_tokenizer = AutoTokenizer.from_pretrained(JUDGE_MODEL_NAME, use_fast=True)
        judge_model = AutoModelForCausalLM.from_pretrained(
            JUDGE_MODEL_NAME,
            device_map="auto",
            torch_dtype="auto",
        )
        judge_model.eval()

    judge_fn = make_llm_judge_fn(
        model=judge_model,
        tokenizer=judge_tokenizer,
        max_candidate_chars=JUDGE_MAX_CANDIDATE_CHARS,
        max_new_tokens=JUDGE_MAX_NEW_TOKENS,
    )
else:
    judge_fn = None
    print("LLM judge disabled because RUN_LLM_JUDGE=False")


Loading separate LLM judge model: Qwen/Qwen2.5-Coder-3B-Instruct


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Evaluate the LLM judge selector on the same `candidate_pool_df` used by the oracle and public-test verifier. This keeps the candidate set fixed so the comparison measures selection quality rather than generation quality.

In [17]:
evaluate_llm_judge_selector = hh.evaluate_llm_judge_selector

if RUN_LLM_JUDGE:
    judge_metrics, judge_selection_df = evaluate_llm_judge_selector(
        problems_df=eval_problems_df,
        pool_df=candidate_pool_df,
        judge_fn=judge_fn,
    )
    print(json.dumps(judge_metrics, indent=2))
    display(judge_selection_df.head())
else:
    judge_metrics = {"method": "llm_judge", "num_tasks": 0, "accuracy": None}
    judge_selection_df = pd.DataFrame()
    print("LLM judge skipped because RUN_LLM_JUDGE=False")


{
  "method": "llm_judge",
  "num_tasks": 50,
  "accuracy": 0.68
}


,task_id,selected_sample_index,selected_passed_full_tests,selected_status,oracle_available,judge_raw_output,judge_reason
0,HumanEval/0,1,False,runtime_error,True,"```json\n{""selected_sample_index"": 1, ""judge_reason"": ""Candidate 1 uses itertools.combinations to efficiently check all pairs without nested loops, which is more concise and avoids potential off-b...","Candidate 1 uses itertools.combinations to efficiently check all pairs without nested loops, which is more concise and avoids potential off-by-one errors."
1,HumanEval/1,0,True,passed,True,"```json\n{""selected_sample_index"": 0, ""judge_reason"": ""Correctly separates groups of nested parentheses without ignoring spaces or using regex.""}\n```",Correctly separates groups of nested parentheses without ignoring spaces or using regex.
2,HumanEval/10,3,False,failed_tests,True,"```json\n{""selected_sample_index"": 3, ""judge_reason"": ""Candidate 3 correctly implements the algorithm to find the shortest palindrome that begins with a supplied string.""}\n```",Candidate 3 correctly implements the algorithm to find the shortest palindrome that begins with a supplied string.
3,HumanEval/11,0,True,passed,True,"```json\n{""selected_sample_index"": 0, ""judge_reason"": ""Correctly implements XOR operation using list comprehension and zip function""}\n```",Correctly implements XOR operation using list comprehension and zip function
4,HumanEval/12,4,True,passed,True,"```json\n{""selected_sample_index"": 4, ""judge_reason"": ""Candidate 4 handles edge cases like empty lists and ties by returning the first longest string, which matches the prompt's requirements.""}\n```","Candidate 4 handles edge cases like empty lists and ties by returning the first longest string, which matches the prompt's requirements."


This table shows what the LLM judge selected after reading the prompt and candidate code.

Useful columns:

- `selected_sample_index`: which candidate the judge chose
- `selected_passed_full_tests`: whether that choice passed the full evaluator
- `oracle_available`: whether any candidate in the pool was actually correct
- `judge_reason`: the judge's short explanation when it could be parsed

A stronger separate judge model may perform better than reusing the generator, but it still does not execute code. For code tasks, executable feedback can remain more reliable than code-reading alone.


## Part 6 - Failure Analysis

This section inspects cases where the verifier helped, missed a correct candidate, or where the whole candidate pool was weak.


In [18]:
build_failure_analysis_table = hh.build_failure_analysis_table

failure_analysis_df = build_failure_analysis_table(
    pool_df=candidate_pool_df,
    public_selections_df=public_selection_df,
    judge_selections_df=judge_selection_df if RUN_LLM_JUDGE else None,
    max_cases=5,
)

display(failure_analysis_df)


,task_id,analysis_label,first_sample_passed,selected_passed_full_tests,oracle_available,public_score,selected_sample_index,judge_passed
0,HumanEval/10,public verifier helped,False,True,True,1.0,2,False
1,HumanEval/25,public verifier helped,False,True,True,1.0,1,True
2,HumanEval/36,public verifier helped,False,True,True,1.0,1,True
3,HumanEval/37,public verifier helped,False,True,True,1.0,4,True
4,HumanEval/39,public verifier helped,False,True,True,1.0,2,True


The selected cases show where verification helped most. In all five rows, the first candidate failed, but the public-test verifier picked a later candidate that passed the full tests.

This is the practical value of test-time compute: generating multiple candidates creates options, and even a weak verifier can sometimes find the good one. The `judge_passed` column also shows that the LLM judge did not always make the same successful choice.


## Final Results

This table compares the baseline, `pass@k`, oracle, public-test verifier, and LLM judge.

The baseline solved `0.68` of tasks, while the frozen candidate pool reached `Pass@5 = 0.80`, meaning extra generation often produced a correct answer somewhere among the five candidates. The public-test verifier selected passing candidates at `0.76`, recovering most of the oracle ceiling and outperforming the LLM judge at `0.68`.

In [19]:
build_results_table = hh.build_results_table

required_results_df = build_results_table(
    baseline=baseline_metrics,
    passk_df=passk_df,
    oracle=oracle_metrics,
    public_metrics=public_metrics,
    judge_metrics=judge_metrics if RUN_LLM_JUDGE else None,
)

display(required_results_df)


,method,accuracy,notes
0,Baseline,0.68,Single generated completion
1,Pass@1,0.58,First sampled candidate from the frozen pool; compare carefully if the pool used different generation settings than the baseline
2,Pass@2,0.68,At least one passing candidate among first k samples
3,Pass@5,0.80,At least one passing candidate among first k samples
4,oracle_best_of_5,0.80,"Selection uses hidden evaluation tests to identify the winning candidate, which is only valid as an upper bound."
5,Public-test verifier,0.76,Candidate chosen using only partial public tests
6,LLM judge,0.68,Candidate chosen by the judge model without hidden tests


### Conclusion

This notebook used verification as a selection step: generate multiple candidates first, then choose one using different signals. The strongest deployable signal here was partial execution on public tests, which was more reliable than asking an LLM judge to read the code without running it.

The practical benefit is that verification can turn extra generation into higher correctness without requiring a larger generator model. It does add compute because multiple candidates must be generated and checked, but it can be cheaper than relying on a much stronger model or manually inspecting outputs. For code tasks, this also helps automate getting correct code: when tests are available, the system can generate options, run lightweight checks, reject obviously broken programs, and select the candidate most likely to pass the full evaluator.


## Save Artifacts

Save the main results for reuse without rerunning the notebook.


In [20]:
final_payload = {
    "config": {
        "model_name": MODEL_NAME,
        "judge_model_name": JUDGE_MODEL_NAME,
        "max_tasks": MAX_TASKS,
        "num_candidates": NUM_CANDIDATES,
        "k_values": K_VALUES,
        "max_new_tokens": MAX_NEW_TOKENS,
        "exec_timeout_seconds": EXEC_TIMEOUT_SECONDS,
        "public_test_fraction": PUBLIC_TEST_FRACTION,
        "run_llm_judge": RUN_LLM_JUDGE,
        "random_seed": RANDOM_SEED,
        "debug_temperature": 0.0,
        "baseline_temperature": 0.0,
        "pool_temperature": 0.8,
        "generation_top_p": GENERATION_TOP_P,
        "judge_max_candidate_chars": JUDGE_MAX_CANDIDATE_CHARS,
        "judge_max_new_tokens": JUDGE_MAX_NEW_TOKENS,
    },
    "baseline_metrics": baseline_metrics,
    "baseline_rows": baseline_rows,
    "pass_at_k": passk_df.to_dict(orient="records"),
    "oracle_metrics": oracle_metrics,
    "public_test_metrics": public_metrics,
    "public_test_rows": public_selection_df.to_dict(orient="records"),
    "judge_metrics": judge_metrics,
    "judge_rows": judge_selection_df.to_dict(orient="records"),
    "failure_analysis": failure_analysis_df.to_dict(orient="records"),
    "required_results": required_results_df.to_dict(orient="records"),
}

results_path = "humaneval_full_assignment_results.json"
with open(results_path, "w", encoding="utf-8") as f:
    json.dump(final_payload, f, indent=2)

print(f"Saved results to: {results_path}")


Saved results to: humaneval_full_assignment_results.json


In [21]:
notebook_end_time = time.time()
total_runtime = notebook_end_time - notebook_start_time

minutes = int(total_runtime // 60)
seconds = int(total_runtime % 60)

print(f"Total notebook runtime: {minutes:02d} minutes and {seconds:02d} seconds")

Total notebook runtime: 14 minutes and 08 seconds
